In [1]:
%cd "E:/src code 2/python 2/KG"
import pandas as pd
import os

from src.index.entity_extractor import EntityExtractor
from src.utils.config_loader import ConfigLoader
from src.llm.gemini import Gemini_LLM
from src.utils.utils import read_file
from src.utils.type import TYPE_OF_ENTITY_IN_KG, TYPE_OF_JOB, TYPE_OF_JOB_ENTITY, TYPE_OF_CV, TYPE_OF_EDGE
from src.db.neo4j import GraphManager, Node, Edge

gm = GraphManager("neo4j://localhost:7687", "neo4j", "123123aA@")
config = ConfigLoader().get_config_from_file(r"E:\src code 2\python 2\Legal_RAG\config\config.yaml")
llm = Gemini_LLM(config=config)

entities = ["Programming Language", "Library", "Software", "Technology", "Task", "Country", "City", "District"]


E:\src code 2\python 2\KG


In [12]:
# df = pd.read_csv("E:/data/cv/UpdatedResumeDataSet.csv")
# df = df[~df["Resume"].str.contains("â")]
# df

In [2]:
folder = "E:/data/cv"
entity_extractor = EntityExtractor(entities=", ".join(entities), llm=llm, cache_folder=folder)

In [14]:
# from src.prompt.index import (
#     GRAPH_EXTRACTION_PROMPT_v0,
#     INCLUDE_RELATIONSHIP_EXTRACTION_PROMPT,
#     CV_EXTRACT_GRAPH_PROMPT,
#     JD_EXTRACT_GRAPH_PROMPT,
#     SUMMERIZE_CV_PROMPT,
#     SUMMERIZE_JD_PROMPT
# )
# for cate in set(df["Category"]):
#     for id, cv in enumerate(df[df["Category"] == cate]['Resume']):
#         response = llm.chat([
#             {'role':'user','content':CV_EXTRACT_GRAPH_PROMPT.format(entity_types=", ".join(entities) ,  input_text= cv)}
            
#         ])
#         with open(f'{folder}/{cate}_{id}.txt', 'w') as f:
#             f.write(response)

           

In [15]:
folder = "E:/data/cv/"
for file in os.listdir(folder):
    if "raw" in file: continue
    with open(folder + file, 'r', encoding="utf-8") as f:
        try:
            data = f.read()
            entities_list, _ = entity_extractor.parse_entities(data)
            list_node = [e.node() for e in entities_list]
            cv_node = Node(TYPE_OF_CV, {"name":file[:-4]})
            for e in list_node:
                e.label = TYPE_OF_JOB_ENTITY
                gm.add_node(e)
                gm.add_edge(Edge(cv_node, e, TYPE_OF_EDGE))
        except:
            print(file)

Java Developer_33.txt
Mechanical Engineer_7.txt
